In [4]:
import pandas as pd
import numpy as np
import csv


In [10]:
path = r"C:\Users\USER\Desktop\DNN project\dnn-project\data\full_set\autotagging_genre.tsv"  # Adjust to your location

with open(path, encoding="utf-8", newline="") as f:
    reader = csv.reader(f, delimiter="\t")
    header = next(reader)

    rows = [
        row[:5] + [[tag.removeprefix("genre---") for tag in row[5:]]]
        for row in reader
        if row
    ]

df_1 = pd.DataFrame(rows, columns=header[:5] + ["genres"])
df_1["DURATION"] = pd.to_numeric(df_1["DURATION"])


In [11]:
path = r"C:\Users\USER\Desktop\DNN project\dnn-project\data\full_set\autotagging_instrument.tsv"  # Adjust to your location

with open(path, encoding="utf-8", newline="") as f:
    reader = csv.reader(f, delimiter="\t")
    header = next(reader)

    rows = [
        row[:5] + [[tag.removeprefix("instrument---") for tag in row[5:]]]
        for row in reader
        if row
    ]

df_2 = pd.DataFrame(rows, columns=header[:5] + ["instruments"])
df_2["DURATION"] = pd.to_numeric(df_2["DURATION"])


In [14]:
df_combined = df_1.merge(
    df_2[["TRACK_ID", "instruments"]],
    on="TRACK_ID",
    how="inner"
)

# Exclude missing or empty genre/instrument lists
df_combined = df_combined[
    df_combined["genres"].str.len().gt(0)
    & df_combined["instruments"].str.len().gt(0)
].reset_index(drop=True)

df_combined.head()

,TRACK_ID,ARTIST_ID,ALBUM_ID,PATH,DURATION,genres,instruments
0,track_0000382,artist_000020,album_000046,82/382.mp3,211.1,[classical],[voice]
1,track_0000383,artist_000020,album_000046,83/383.mp3,113.1,[classical],[voice]
2,track_0000384,artist_000020,album_000046,84/384.mp3,115.7,[classical],[voice]
3,track_0000386,artist_000020,album_000046,86/386.mp3,103.4,[classical],[voice]
4,track_0000387,artist_000020,album_000046,87/387.mp3,257.1,[classical],[voice]


In [15]:
df_combined.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24894 entries, 0 to 24893
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   TRACK_ID     24894 non-null  object 
 1   ARTIST_ID    24894 non-null  object 
 2   ALBUM_ID     24894 non-null  object 
 3   PATH         24894 non-null  object 
 4   DURATION     24894 non-null  float64
 5   genres       24894 non-null  object 
 6   instruments  24894 non-null  object 
dtypes: float64(1), object(6)
memory usage: 1.3+ MB


In [20]:
genre_counts = (
    df_combined["genres"]
    .explode()
    .value_counts()
    .rename_axis("genre")
    .reset_index(name="track_count")
)

genre_counts.head(40)

,genre,track_count
0,electronic,5572
1,soundtrack,4441
2,ambient,4005
3,classical,3812
4,pop,3305
5,rock,3203
6,chillout,2258
7,easylistening,2147
8,jazz,1625
9,experimental,1559


In [21]:
selected_genres = [
    "classical", "rock", "electronic", "jazz", "folk", "hiphop"
]

df_selected = df_combined.copy()
df_selected["target_genres"] = df_selected["genres"].apply(
    lambda genres: [g for g in genres if g in selected_genres]
)

df_selected = df_selected[
    df_selected["target_genres"].str.len().gt(0)
].reset_index(drop=True)

In [22]:
df_selected.head()

,TRACK_ID,ARTIST_ID,ALBUM_ID,PATH,DURATION,genres,instruments,target_genres
0,track_0000382,artist_000020,album_000046,82/382.mp3,211.1,[classical],[voice],[classical]
1,track_0000383,artist_000020,album_000046,83/383.mp3,113.1,[classical],[voice],[classical]
2,track_0000384,artist_000020,album_000046,84/384.mp3,115.7,[classical],[voice],[classical]
3,track_0000386,artist_000020,album_000046,86/386.mp3,103.4,[classical],[voice],[classical]
4,track_0000387,artist_000020,album_000046,87/387.mp3,257.1,[classical],[voice],[classical]


In [23]:
df = df_combined.drop_duplicates("TRACK_ID").copy()

df["target_genres"] = df["genres"].apply(
    lambda genres: [g for g in selected_genres if g in genres]
)

# Find how many tracks are available for each genre
counts = (
    df["target_genres"]
    .explode()
    .value_counts()
    .reindex(selected_genres, fill_value=0)
)

# Use the smallest genre's count as the sampling target
n_per_genre = int(counts.min())
if n_per_genre == 0:
    raise ValueError("At least one selected genre has no tracks.")

samples = [
    df[df["target_genres"].apply(lambda tags: genre in tags)]
    .sample(n=n_per_genre, random_state=42)
    for genre in selected_genres
]

split_df = (
    pd.concat(samples)
    .drop_duplicates("TRACK_ID")
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

# Save lists as JSON strings inside the CSV
import json

csv_df = split_df.copy()
for column in ["genres", "instruments", "target_genres"]:
    csv_df[column] = csv_df[column].apply(json.dumps)

csv_df.to_csv("split_csv.csv", index=False)

print(f"Saved {len(split_df)} unique tracks")
print(split_df["target_genres"].explode().value_counts())

Saved 7324 unique tracks
target_genres
electronic    1733
rock          1486
classical     1446
jazz          1376
folk          1323
hiphop        1304
Name: count, dtype: int64
